This notebook scrapes data from yfinance, using the NumPy, pandas, and scipy libraries computes descriptive statistics and the four moments.

We begin by using yf.download to call the adjusted closing prices of the given securities. We use SPY instead of ^GSPC because SPY is actually tradeable.


In [11]:
import pandas as pd
import numpy as np
from scipy.stats import skew
from scipy.stats import kurtosis
from scipy.stats import norm
import matplotlib.pyplot as plt
yf.pdr_override()

## Return Definitions
Simple returns are intuitive and used for interpretation and reporting.<br>

Log returns are additive over time and more convenient for modelling and statistical analysis.
Both are computed here to compare their behaviour on daily equity data.

In [12]:
#import yfinance for data
import yfinance as yf
secs = ['AAPL', 'SAIL.NS', 'MSFT', 'RELIANCE.NS', 'SPY'] #define securities in a list
data = yf.download(secs, start='2016-01-01', end='2026-01-01', auto_adjust=False) #download data within time frame, auto_adjust false
#means adj close is available separately, otherwise OHLC is adjusted by default
adj_close = data['Adj Close'].dropna() #defining adjusted close and dropna removes N/A entries

returns = adj_close.pct_change().dropna() #computing returns cause prices are usless and dropna removes N/A entries
log_returns = np.log(adj_close).diff().dropna() #computing log returns
print(returns.head())
print(log_returns.head())

Failed to get ticker 'MSFT' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'SAIL.NS' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'SPY' reason: Expecting value: line 1 column 1 (char 0)
[                       0%%                      ]Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'RELIANCE.NS' reason: Expecting value: line 1 column 1 (char 0)
[*********************100%%**********************]  4 of 5 completed

5 Failed downloads:
[*********************100%%**********************]  4 of 5 completed['MSFT', 'SAIL.NS', 'SPY', 'AAPL', 'RELIANCE.NS']: Exception('%ticker%: No timezone found, symbol may be delisted')


ValueError: attempt to get argmax of an empty sequence

From the prices, we compute simple returns. Analysis of returns leads to data that can be studied. Direct analysis of prices would lead to nonsensical results. We compute descriptive statistics for each asset seprately.

In [ ]:
# Now calculate statistics PER ASSET (column-wise).
print("=== Means (Annualised Daily Return) ===")
print(returns.mean()*252)
print(log_returns.mean()*252)
mean = returns.mean()*252
mean2 = log_returns.mean()*252
print("\n=== Standard Deviation (Annualised Daily Volatility) ===") # \n is like text break
std = returns.std()*np.sqrt(252)
std2 = log_returns.std()*np.sqrt(252)
print(returns.std()*np.sqrt(252))
risk_free_rate = 0.02  # 2% annual

# Skew and Kurtosis
print("\n=== Skewness ===")
print(returns.skew())
print("\n=== Kurtosis (Excess) ===")
print(returns.kurtosis())

Plotting Sharpe ratios, means, and standard deviations

In [ ]:
plt.plot(std)
plt.title('Annualized Standard Deviations')
plt.xlabel('Asset')
plt.ylabel('Standard Deviation')
plt.show()

plt.plot(mean)
plt.title('Annualized Means')
plt.xlabel('Asset')
plt.ylabel('Mean')
plt.show()

plt.plot(std2)
plt.title('Annualized Standard Deviation 2')
plt.xlabel('Asset')
plt.ylabel('Standard Deviation 2')
plt.show()

plt.plot(mean2)
plt.title('Annualized Means 2')
plt.xlabel('Asset')
plt.ylabel('Mean 2')
plt.show()

Now, we use a correlation matrix to study dependencies across our porfolio. First, we compute a returns correlation matrix. Then we compute annualised volatilities and finally, annualised covariance matrix.

In [ ]:
# 1. Correlation matrix (from daily returns)
corr_matrix = returns.corr()
cr_mat = log_returns.corr()
print("Correlation Matrix (Daily):")
print(corr_matrix.round(3))
print("Correlation Matrix Log Returns (Daily):")
print(cr_mat.round(3))

In [ ]:
# 1. Calculate annualized volatilities
annual_vols = returns.std() * np.sqrt(252)  # This is your 'vols'
annual_vols2 = log_returns.std() * np.sqrt(252)  # This is your 'vols'

# 2. Correlation matrix (from daily returns)
corr_matrix = returns.corr()
corr_matrix2 = log_returns.corr()

# 3. Build the annualized covariance matrix
#    Σ_annual = Corr * (σ_i * σ_j) for all i,j
cov_matrix_annual = corr_matrix * np.outer(annual_vols, annual_vols) #computes the outer product of two vectors
cov_matrix_annual2 = corr_matrix2 * np.outer(annual_vols2, annual_vols2) #

In [ ]:
# 2. Annualized covariance matrix: Corr * (std_i * std_j)
# You already have annualized volatilities (std_annual)
std_annual = returns.std() * np.sqrt(252)  # Should match your earlier numbers
std_annual

In [ ]:
std_annual2 = log_returns.std() * np.sqrt(252)  # Should match your earlier numbers
std_annual2

In [ ]:
# Build the covariance matrix manually to understand it:
cov_matrix_annual = np.outer(std_annual, std_annual) * corr_matrix
cov_matrix_annual2 = np.outer(std_annual2, std_annual2) * corr_matrix2

In [ ]:
print("\nAnnualized Covariance Matrix 2:")
print(pd.DataFrame(cov_matrix_annual2, index=std_annual.index, columns=std_annual.index).round(6))

Calculating beta and CAPM

In [ ]:
# 1. Beta & CAPM Expected Return
market_returns = returns['SPY']
market_returns2 = log_returns['SPY']
stock_returns = returns.drop(columns=['SPY'])
stock_log_returns = log_returns.drop(columns=['SPY'])
betas = {}
for stock in stock_returns.columns:
    cov = np.cov(stock_returns[stock], market_returns)[0,1]
    beta = cov / np.var(market_returns)
    betas[stock] = beta
    # Expected Return = Rf + beta * (Market Return - Rf)
    expected_return = risk_free_rate + beta * (market_returns.mean() * 252 - risk_free_rate)
    print(f"{stock}: Beta = {beta:.3f}, Exp Return = {expected_return:.2%}")

for stck in stock_log_returns:
    cov = np.cov(stock_log_returns[stck], market_returns2)[0,1]
    beta2 = cov / np.var(market_returns2)
    betas[stck] = beta2
    # Expected Return = Rf + beta * (Market Return - Rf)
    expected_return = risk_free_rate + beta2 * (market_returns2.mean() * 252 - risk_free_rate)
    print(f"{stck}: Beta = {beta2:.3f}, Exp Return = {expected_return:.2%}")

In [ ]:
# 1. Calculate annualized volatilities
annual_vols = returns.std() * np.sqrt(252)  # This is your 'vols'

# 2. Correlation matrix (from daily returns)
corr_matrix = returns.corr()

# 3. Build the annualized covariance matrix
#    Σ_annual = Corr * (σ_i * σ_j) for all i,j
cov_matrix_annual = corr_matrix * np.outer(annual_vols, annual_vols)
cov_matrix_annual

In [ ]:
print("AAPL annual vol:", std_annual['AAPL'])           # Should be ~0.296
print("AAPL annual variance:", std_annual['AAPL']**2)   # Should be ~0.0876
print("Cov matrix diagonal:", cov_matrix_annual.loc['AAPL', 'AAPL'])  # Should match ^

In [ ]:
# 4. Verify shape and print a snippet
print("Covariance Matrix (annualized) shape:", cov_matrix_annual.shape)
print("\nDiagonal elements (should be variances, i.e., vol^2):")
for i, ticker in enumerate(returns.columns):
    # Use .iloc for integer-based indexing on the DataFrame
    print(f"{ticker}: {cov_matrix_annual.iloc[i, i]:.6f} ≈ {annual_vols[ticker]**2:.6f}")

In [ ]:
# 5. Define portfolio weights (equal-weight example)
assets = returns.columns.tolist()
weights = np.array([1/len(assets)] * len(assets))  # Equal weight

# 6. Portfolio variance & volatility (annualized)
portfolio_variance = weights.T @ cov_matrix_annual @ weights
portfolio_vol = np.sqrt(portfolio_variance)

#7. Portfolio returns
portfolio_returns = returns @ weights

print(f"\nEqual-weight portfolio annual volatility: {portfolio_vol:.4f} ({portfolio_vol*100:.2f}%)")

In [ ]:
vol_values = [0.296413, 0.273108, 0.273140, 0.449830, 0.181682]
np.mean(vol_values)

The earlier individual volatilities were 0.2964, 0.2731, 0.2731, 0.4498, 0.1817, meaning average is 0.295, however, equal-weighted annual vol is 0.197. <br> <br> That is a 33.22% reduction in volatility.




In [ ]:
# Now we compute VaR on our portfolio
# Regulatory market risk frameworks commonly require 10-day risk measures
# Therefore, 1-day VaR is scaled using the square-root-of-time rule.

alpha_var = 0.95
alpha_es  = 0.975

var_1d = np.percentile(portfolio_returns, (1 - alpha_var) * 100)
var_10d_loss = -var_1d*np.sqrt(10)
es_var_threshold = np.percentile(portfolio_returns, (1 - alpha_es) * 100)
tail_returns = portfolio_returns[portfolio_returns <= es_var_threshold]
es_10d_loss = -tail_returns.mean()*np.sqrt(10)

assert var_10d_loss > 0, "VaR loss must be positive"
assert es_10d_loss > 0, "ES loss must be positive"
assert es_10d_loss >= var_10d_loss, "ES should be greater than or equal to VaR"

In [ ]:
print(f"10-day {alpha_var*100: .0f}% Historical VaR: {var_10d_loss:.2%}")
print(f"10-day {alpha_es*100:.1f}% Historical ES: {es_10d_loss:.2%}")

In [ ]:
mu = portfolio_returns.mean()
sigma = portfolio_returns.std()
z_var = norm.ppf(1 - alpha_var)
var_parametric_10d_loss = -(mu + sigma * z_var)*np.sqrt(10)

z_es = norm.ppf(1 - alpha_es)
es_parametric_10d_loss = (mu + sigma * norm.pdf(z_es) / (1 - alpha_es))*np.sqrt(10)

assert var_parametric_10d_loss > 0, "Parametric VaR must be positive"
assert es_parametric_10d_loss > 0, "Parametric ES must be positive"
assert es_parametric_10d_loss >= var_parametric_10d_loss, "ES must be >= VaR"

In [ ]:
print(f"10-day {alpha_var*100: .0f}% Parametric VaR (Normal): {var_parametric_10d_loss:.2%}")
print(f"10-day {alpha_es*100}% Parametric ES (Normal): {es_parametric_10d_loss:.2%}")

Parametric risk measures rely on distributional assumptions and may underestimate tail risk in the presence of skewness and excess kurtosis.<br><br>
Parametric ES increases more than √T due to tail sensitivity of φ(z)/(1−α) under the normal assumption

### Time Scaling of Parametric Expected Shortfall

While VaR and ES are often approximated as scaling with √T under
Gaussian i.i.d. assumptions, parametric ES does not scale exactly
by √T.

The Expected Shortfall formula is:

ES_α = μ + σ · φ(z_α) / (1 − α)

When scaling from 1 day to T days:

μ_T = Tμ  
σ_T = √T σ

Thus:

ES_T = Tμ + √T σ · φ(z_α) / (1 − α)

Unless μ = 0, the ratio ES_T / ES_1d is not exactly √T.
At high confidence levels, the ES tail factor amplifies this effect,
causing parametric ES to grow slightly faster than √T.

This explains why 10-day parametric ES exceeds the naive √10 scaling
of the 1-day ES.